In [ ]:
!pip install sentence-transformers pandas -q

In [ ]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import random
import re
from datetime import datetime

모델 로드

In [ ]:
model = SentenceTransformer("jhgan/ko-sroberta-multitask")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임시 db 테이블

In [ ]:
users = []
audio_records = []
speech_analysis_results = []
text_analysis_results = []
recall_questions = []
recall_answers = []
recall_keywords = []
recall_analysis_results = []
risk_analysis_results = []

공통함수

In [ ]:
def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def clean_text(text):
    text = str(text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", "", text)
    return text.strip()

def next_id(table):
    return len(table) + 1

사용자 생성

In [ ]:
def create_user(name, birth_year):
    user = {
        "user_id": next_id(users),
        "name": name,
        "birth_year": birth_year,
        "created_at": now()
    }
    users.append(user)
    return user

user = create_user("김경빈", 2000)
user

{'user_id': 1,
 'name': '김경빈',
 'birth_year': 2000,
 'created_at': '2026-05-12 21:13:44'}

회상질문 저장

In [ ]:
def create_recall_question(user_id, question_text, question_type, category, keywords):
    question_id = next_id(recall_questions)

    question = {
        "recall_question_id": question_id,
        "user_id": user_id,
        "question_text": question_text,
        "question_type": question_type,
        "category": category,
        "created_at": now()
    }

    recall_questions.append(question)

    for keyword in keywords:
        recall_keywords.append({
            "keyword_id": next_id(recall_keywords),
            "recall_question_id": question_id,
            "keyword_text": keyword
        })

    return question

초기 질문

In [ ]:
def create_recall_question(user_id, question_text, question_type, category, keywords):
    question_id = next_id(recall_questions)

    question = {
        "recall_question_id": question_id,
        "user_id": user_id,
        "question_text": question_text,
        "question_type": question_type,
        "category": category,
        "created_at": now()
    }

    recall_questions.append(question)

    for keyword in keywords:
        recall_keywords.append({
            "keyword_id": next_id(recall_keywords),
            "recall_question_id": question_id,
            "keyword_text": keyword
        })

    return question

audio_records 저장 함수

In [ ]:
def create_audio_record(user_id, transcript_text, audio_duration=None, audio_file_path=None, stt_confidence=None):
    record_id = next_id(audio_records)

    if audio_duration is None:
        audio_duration = round(random.uniform(5.0, 40.0), 2)

    if audio_file_path is None:
        audio_file_path = f"/audio/user{user_id}/record_{record_id:03d}.wav"

    record = {
        "record_id": record_id,
        "user_id": user_id,
        "audio_file_path": audio_file_path,
        "audio_duration": audio_duration,
        "transcript_text": transcript_text,
        "stt_confidence": stt_confidence,
        "recorded_at": now()
    }

    audio_records.append(record)
    return record

recall_answers 저장 함수

In [ ]:
def create_recall_answer(user_id, recall_question_id, answer_type, transcript_text):
    audio_record = create_audio_record(
        user_id=user_id,
        transcript_text=transcript_text,
        stt_confidence=round(random.uniform(0.85, 0.99), 2)
    )

    answer = {
        "recall_answer_id": next_id(recall_answers),
        "recall_question_id": recall_question_id,
        "user_id": user_id,
        "record_id": audio_record["record_id"],
        "answer_type": answer_type,
        "created_at": now()
    }

    recall_answers.append(answer)
    return answer

초기 답변 저장

In [ ]:
create_recall_answer(1, 1, "INITIAL", "제 고향은 대구입니다.")
create_recall_answer(1, 2, "INITIAL", "김치찌개를 가장 좋아합니다.")
create_recall_answer(1, 3, "INITIAL", "가족들과 제주도 여행을 갔던 기억이 가장 좋았습니다.")
create_recall_answer(1, 4, "INITIAL", "오늘 아침에는 미역국과 밥을 먹었습니다.")

pd.DataFrame(recall_answers)

,recall_answer_id,recall_question_id,user_id,record_id,answer_type,created_at
0,1,1,1,1,INITIAL,2026-05-12 21:13:44
1,2,2,1,2,INITIAL,2026-05-12 21:13:44
2,3,3,1,3,INITIAL,2026-05-12 21:13:44
3,4,4,1,4,INITIAL,2026-05-12 21:13:44


현재 회상 답변 저장

In [ ]:
create_recall_answer(1, 1, "INITIAL", "제 고향은 대구입니다.")
create_recall_answer(1, 2, "INITIAL", "김치찌개를 가장 좋아합니다.")
create_recall_answer(1, 3, "INITIAL", "가족들과 제주도 여행을 갔던 기억이 가장 좋았습니다.")
create_recall_answer(1, 4, "INITIAL", "오늘 아침에는 미역국과 밥을 먹었습니다.")

pd.DataFrame(recall_answers)

,recall_answer_id,recall_question_id,user_id,record_id,answer_type,created_at
0,1,1,1,1,INITIAL,2026-05-12 21:13:44
1,2,2,1,2,INITIAL,2026-05-12 21:13:44
2,3,3,1,3,INITIAL,2026-05-12 21:13:44
3,4,4,1,4,INITIAL,2026-05-12 21:13:44
4,5,1,1,5,INITIAL,2026-05-12 21:13:44
5,6,2,1,6,INITIAL,2026-05-12 21:13:44
6,7,3,1,7,INITIAL,2026-05-12 21:13:44
7,8,4,1,8,INITIAL,2026-05-12 21:13:44


record_id로 STT 텍스트 찾기

In [ ]:
def get_transcript_by_record_id(record_id):
    for record in audio_records:
        if record["record_id"] == record_id:
            return record["transcript_text"]
    return None

def get_question_by_id(question_id):
    for q in recall_questions:
        if q["recall_question_id"] == question_id:
            return q
    return None

def get_keywords_by_question_id(question_id):
    return [
        k["keyword_text"]
        for k in recall_keywords
        if k["recall_question_id"] == question_id
    ]

음성 feature 분석

In [ ]:
def analyze_speech(record_id):
    result = {
        "speech_analysis_id": next_id(speech_analysis_results),
        "record_id": record_id,
        "pause_count": random.randint(1, 10),
        "total_pause_duration": round(random.uniform(1.0, 12.0), 2),
        "avg_pause_duration": round(random.uniform(0.2, 2.0), 2),
        "max_pause_duration": round(random.uniform(1.0, 5.0), 2),
        "rms_mean": round(random.uniform(0.01, 0.1), 4),
        "rms_std": round(random.uniform(0.001, 0.05), 4),
        "zcr_mean": round(random.uniform(0.01, 0.2), 4),
        "zcr_std": round(random.uniform(0.001, 0.05), 4),
        "spectral_centroid_mean": round(random.uniform(1000, 4000), 2),
        "spectral_centroid_std": round(random.uniform(100, 800), 2),
        "mfcc_1_mean": round(random.uniform(-300, 100), 2),
        "mfcc_2_mean": round(random.uniform(-50, 50), 2),
        "mfcc_3_mean": round(random.uniform(-50, 50), 2),
        "mfcc_4_mean": round(random.uniform(-50, 50), 2),
        "mfcc_5_mean": round(random.uniform(-50, 50), 2),
        "speech_rate": round(random.uniform(2.0, 5.0), 2),
        "articulation_score": random.randint(50, 100),
        "pronunciation_stability": random.randint(50, 100),
        "response_latency": round(random.uniform(0.5, 5.0), 2),
        "repetition_count": random.randint(0, 5),
        "filler_count": random.randint(0, 8),
        "analyzed_at": now()
    }

    speech_analysis_results.append(result)
    return result

텍스트 feature 분석

In [ ]:
def analyze_text(record_id):
    text = get_transcript_by_record_id(record_id)
    text = clean_text(text)

    words = text.split()
    word_count = len(words)
    sentence_count = max(1, text.count(".") + text.count("?") + text.count("!"))

    unique_words = set(words)
    lexical_diversity = len(unique_words) / word_count if word_count > 0 else 0

    repeated_word_ratio = 1 - lexical_diversity if word_count > 0 else 0

    result = {
        "text_analysis_id": next_id(text_analysis_results),
        "record_id": record_id,
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_sentence_length": round(word_count / sentence_count, 2),
        "lexical_diversity": round(lexical_diversity, 2),
        "repeated_word_ratio": round(repeated_word_ratio, 2),
        "incomplete_sentence_count": random.randint(0, 2),
        "topic_coherence_score": round(random.uniform(0.5, 1.0), 2),
        "analyzed_at": now()
    }

    text_analysis_results.append(result)
    return result

질문 유형별 가중치

In [ ]:
def get_recall_weights(question_type):
    weights = {
        "FACT": (0.2, 0.8),
        "PREFERENCE": (0.4, 0.6),
        "MEMORY": (0.6, 0.4),
        "DAILY": (0.3, 0.7)
    }
    return weights.get(question_type, (0.4, 0.6))

회상 일치도 분석

In [ ]:
def calculate_similarity_score(past_text, current_text):
    emb1 = model.encode(clean_text(past_text), convert_to_tensor=True)
    emb2 = model.encode(clean_text(current_text), convert_to_tensor=True)
    score = util.cos_sim(emb1, emb2).item()
    return max(0, score * 100)

def calculate_keyword_score(keywords, current_text):
    current_text = clean_text(current_text)

    if not keywords:
        return 0

    matched = 0
    for keyword in keywords:
        if keyword in current_text:
            matched += 1

    return matched / len(keywords) * 100

문항별 회상 분석 실행

In [ ]:
def analyze_recall_for_question(user_id, question_id):
    question = get_question_by_id(question_id)
    keywords = get_keywords_by_question_id(question_id)

    initial_answer = None
    recall_answer = None

    for ans in recall_answers:
        if ans["user_id"] == user_id and ans["recall_question_id"] == question_id:
            if ans["answer_type"] == "INITIAL":
                initial_answer = ans
            elif ans["answer_type"] == "RECALL":
                recall_answer = ans

    if initial_answer is None or recall_answer is None:
        return None

    past_text = get_transcript_by_record_id(initial_answer["record_id"])
    current_text = get_transcript_by_record_id(recall_answer["record_id"])

    similarity_score = calculate_similarity_score(past_text, current_text)
    keyword_score = calculate_keyword_score(keywords, current_text)

    sim_weight, key_weight = get_recall_weights(question["question_type"])

    final_recall_score = similarity_score * sim_weight + keyword_score * key_weight

    if question["question_type"] == "FACT" and keyword_score == 100:
        final_recall_score = max(final_recall_score, 90)

    if question["question_type"] == "FACT" and keyword_score == 0:
        final_recall_score = min(final_recall_score, 49)

    result = {
        "recall_result_id": next_id(recall_analysis_results),
        "recall_question_id": question_id,
        "past_answer_id": initial_answer["recall_answer_id"],
        "current_answer_id": recall_answer["recall_answer_id"],
        "similarity_score": round(similarity_score, 2),
        "keyword_score": round(keyword_score, 2),
        "final_recall_score": round(final_recall_score, 2),
        "analyzed_at": now()
    }

    recall_analysis_results.append(result)
    return result

모든 회상 질문 분석

In [ ]:
for q in recall_questions:
    analyze_recall_for_question(1, q["recall_question_id"])

pd.DataFrame(recall_analysis_results)

""


모든 음성/텍스트 분석 실행

In [ ]:
for record in audio_records:
    analyze_speech(record["record_id"])
    analyze_text(record["record_id"])

pd.DataFrame(speech_analysis_results).head()

,speech_analysis_id,record_id,pause_count,total_pause_duration,avg_pause_duration,max_pause_duration,rms_mean,rms_std,zcr_mean,zcr_std,...,mfcc_3_mean,mfcc_4_mean,mfcc_5_mean,speech_rate,articulation_score,pronunciation_stability,response_latency,repetition_count,filler_count,analyzed_at
0,1,1,6,9.32,1.11,1.62,0.0978,0.0495,0.0916,0.0370,...,-8.89,-2.99,-24.55,3.54,56,73,0.90,5,3,2026-05-12 21:13:44
1,2,2,1,8.60,1.45,4.15,0.0381,0.0396,0.0928,0.0250,...,-22.88,40.20,3.65,2.75,76,75,3.82,2,7,2026-05-12 21:13:44
2,3,3,5,4.96,0.96,3.18,0.0342,0.0096,0.1864,0.0016,...,-23.88,10.20,-29.75,3.96,93,68,2.75,2,6,2026-05-12 21:13:44
3,4,4,2,10.00,0.24,2.49,0.0699,0.0132,0.0348,0.0223,...,7.36,-43.36,-13.75,3.60,70,71,2.98,1,4,2026-05-12 21:13:44
4,5,5,1,7.43,0.96,1.82,0.0307,0.0106,0.0579,0.0286,...,22.60,18.68,-18.18,4.90,51,94,0.79,0,1,2026-05-12 21:13:44


점수 계산 함수

In [ ]:
def calculate_speech_score(record_id):
    result = None

    for r in speech_analysis_results:
        if r["record_id"] == record_id:
            result = r
            break

    if result is None:
        return None

    articulation = result["articulation_score"]
    stability = result["pronunciation_stability"]

    # filler, pause, latency가 높을수록 감점
    penalty = (
        result["pause_count"] * 1.5
        + result["filler_count"] * 2
        + result["response_latency"] * 2
    )

    score = (articulation * 0.5 + stability * 0.5) - penalty

    return max(0, min(100, score))

def calculate_text_score(record_id):
    result = None

    for r in text_analysis_results:
        if r["record_id"] == record_id:
            result = r
            break

    if result is None:
        return None

    lexical_score = result["lexical_diversity"] * 100
    coherence_score = result["topic_coherence_score"] * 100
    repetition_penalty = result["repeated_word_ratio"] * 30
    incomplete_penalty = result["incomplete_sentence_count"] * 5

    score = (lexical_score * 0.4 + coherence_score * 0.6) - repetition_penalty - incomplete_penalty

    return max(0, min(100, score))

def calculate_recall_score(user_id):
    user_question_ids = [
        q["recall_question_id"]
        for q in recall_questions
        if q["user_id"] == user_id
    ]

    scores = [
        r["final_recall_score"]
        for r in recall_analysis_results
        if r["recall_question_id"] in user_question_ids
    ]

    if not scores:
        return None

    return sum(scores) / len(scores)

최종 위험도 계산

In [ ]:
def calculate_final_risk(user_id):
    user_records = [
        r for r in audio_records
        if r["user_id"] == user_id
    ]

    speech_scores = []
    text_scores = []

    for record in user_records:
        s_score = calculate_speech_score(record["record_id"])
        t_score = calculate_text_score(record["record_id"])

        if s_score is not None:
            speech_scores.append(s_score)

        if t_score is not None:
            text_scores.append(t_score)

    speech_score = sum(speech_scores) / len(speech_scores) if speech_scores else 0
    text_score = sum(text_scores) / len(text_scores) if text_scores else 0
    recall_score = calculate_recall_score(user_id)

    if recall_score is None:
        recall_score = 0

    # 정상 점수가 높을수록 양호
    health_score = (
        speech_score * 0.3
        + text_score * 0.2
        + recall_score * 0.5
    )

    # 위험도 점수는 높을수록 위험
    final_risk_score = 100 - health_score

    if final_risk_score < 30:
        risk_level = "low"
    elif final_risk_score < 60:
        risk_level = "medium"
    else:
        risk_level = "high"

    result = {
        "risk_result_id": next_id(risk_analysis_results),
        "user_id": user_id,
        "record_id": user_records[-1]["record_id"],
        "speech_score": round(speech_score, 2),
        "text_score": round(text_score, 2),
        "recall_score": round(recall_score, 2),
        "final_risk_score": round(final_risk_score, 2),
        "risk_level": risk_level,
        "analyzed_at": now()
    }

    risk_analysis_results.append(result)
    return result

calculate_final_risk(1)

{'risk_result_id': 1,
 'user_id': 1,
 'record_id': 8,
 'speech_score': 54.51,
 'text_score': 77.9,
 'recall_score': 0,
 'final_risk_score': 68.07,
 'risk_level': 'high',
 'analyzed_at': '2026-05-12 21:13:44'}

최종 결과 보기

In [ ]:
print("users")
display(pd.DataFrame(users))

print("audio_records")
display(pd.DataFrame(audio_records))

print("recall_questions")
display(pd.DataFrame(recall_questions))

print("recall_answers")
display(pd.DataFrame(recall_answers))

print("recall_analysis_results")
display(pd.DataFrame(recall_analysis_results))

print("risk_analysis_results")
display(pd.DataFrame(risk_analysis_results))

users


,user_id,name,birth_year,created_at
0,1,김경빈,2000,2026-05-12 21:13:44


audio_records


,record_id,user_id,audio_file_path,audio_duration,transcript_text,stt_confidence,recorded_at
0,1,1,/audio/user1/record_001.wav,18.31,제 고향은 대구입니다.,0.87,2026-05-12 21:13:44
1,2,1,/audio/user1/record_002.wav,26.80,김치찌개를 가장 좋아합니다.,0.94,2026-05-12 21:13:44
2,3,1,/audio/user1/record_003.wav,14.48,가족들과 제주도 여행을 갔던 기억이 가장 좋았습니다.,0.90,2026-05-12 21:13:44
3,4,1,/audio/user1/record_004.wav,11.09,오늘 아침에는 미역국과 밥을 먹었습니다.,0.90,2026-05-12 21:13:44
4,5,1,/audio/user1/record_005.wav,12.80,제 고향은 대구입니다.,0.90,2026-05-12 21:13:44
5,6,1,/audio/user1/record_006.wav,19.73,김치찌개를 가장 좋아합니다.,0.97,2026-05-12 21:13:44
6,7,1,/audio/user1/record_007.wav,26.57,가족들과 제주도 여행을 갔던 기억이 가장 좋았습니다.,0.88,2026-05-12 21:13:44
7,8,1,/audio/user1/record_008.wav,5.36,오늘 아침에는 미역국과 밥을 먹었습니다.,0.93,2026-05-12 21:13:44


recall_questions


...

,recall_answer_id,recall_question_id,user_id,record_id,answer_type,created_at
0,1,1,1,1,INITIAL,2026-05-12 21:13:44
1,2,2,1,2,INITIAL,2026-05-12 21:13:44
2,3,3,1,3,INITIAL,2026-05-12 21:13:44
3,4,4,1,4,INITIAL,2026-05-12 21:13:44
4,5,1,1,5,INITIAL,2026-05-12 21:13:44
5,6,2,1,6,INITIAL,2026-05-12 21:13:44
6,7,3,1,7,INITIAL,2026-05-12 21:13:44
7,8,4,1,8,INITIAL,2026-05-12 21:13:44


recall_analysis_results


""


risk_analysis_results


,risk_result_id,user_id,record_id,speech_score,text_score,recall_score,final_risk_score,risk_level,analyzed_at
0,1,1,8,54.51,77.9,0,68.07,high,2026-05-12 21:13:44


문항별 회상 결과 보기 좋게 출력

In [ ]:
summary_rows = []

for result in recall_analysis_results:
    question = get_question_by_id(result["recall_question_id"])

    past_answer = None
    current_answer = None

    for ans in recall_answers:
        if ans["recall_answer_id"] == result["past_answer_id"]:
            past_answer = ans
        if ans["recall_answer_id"] == result["current_answer_id"]:
            current_answer = ans

    summary_rows.append({
        "질문": question["question_text"],
        "질문유형": question["question_type"],
        "기준답변": get_transcript_by_record_id(past_answer["record_id"]),
        "현재답변": get_transcript_by_record_id(current_answer["record_id"]),
        "의미유사도": result["similarity_score"],
        "키워드점수": result["keyword_score"],
        "최종회상점수": result["final_recall_score"]
    })

pd.DataFrame(summary_rows)

""


In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['34E03CF4-5BDA-471A-B492-9B1A1F03F9B7.png', '7BEC2669-49E9-4327-8ED5-8347F8963346.png', 'Colab Notebooks', 'mbti_model.zip', 'Korean-MBTI-Conversation-Dataset-main.zip', 'MBTI 500.csv', 'ei_model_koelectra_final7_best_accuracy', 'ns_model_koelectra_final7_best_accuracy', 'tf_model_koelectra_final7_best_accuracy', 'pj_model_koelectra_final7_best_accuracy', 'tf_model_koelectra_final8_best_accuracy', '타이타닉데이터셋.csv', 'tripadvisor_hotel_reviews.csv', 'pre_proc_TripAdvisor.csv', 'midas_recall_label_dataset_100.csv']


In [ ]:
import pandas as pd

csv_path = "/content/drive/MyDrive/midas_recall_label_dataset_100.csv"

df = pd.read_csv(csv_path)
df.head()

,id,question_type,category,question,expected_answer,past_answer,current_answer,label
0,1,PROFILE,NAME,성함이 어떻게 되시나요?,김영희,김영희,김영희입니다,일치
1,2,PROFILE,NAME,성함이 어떻게 되시나요?,김영희,김영희,영희요,부분일치
2,3,PROFILE,NAME,성함이 어떻게 되시나요?,김영희,김영희,박영희입니다,부분일치
3,4,PROFILE,NAME,성함이 어떻게 되시나요?,김영희,김영희,이순자입니다,불일치
4,5,PROFILE,NAME,성함이 어떻게 되시나요?,김영희,김영희,잘 모르겠어요,무응답


In [ ]:
def make_input_text(row):
    return (
        f"질문: {row['question']} "
        f"[SEP] 기준답변: {row['expected_answer']} "
        f"[SEP] 이전답변: {row['past_answer']} "
        f"[SEP] 현재답변: {row['current_answer']}"
    )

df["text"] = df.apply(make_input_text, axis=1)

label2id = {
    "일치": 0,
    "부분일치": 1,
    "불일치": 2,
    "무응답": 3
}

id2label = {v: k for k, v in label2id.items()}
df["labels"] = df["label"].map(label2id)

df[["text", "label", "labels"]].head()

,text,label,labels
0,질문: 성함이 어떻게 되시나요? [SEP] 기준답변: 김영희 [SEP] 이전답변: ...,일치,0
1,질문: 성함이 어떻게 되시나요? [SEP] 기준답변: 김영희 [SEP] 이전답변: ...,부분일치,1
2,질문: 성함이 어떻게 되시나요? [SEP] 기준답변: 김영희 [SEP] 이전답변: ...,부분일치,1
3,질문: 성함이 어떻게 되시나요? [SEP] 기준답변: 김영희 [SEP] 이전답변: ...,불일치,2
4,질문: 성함이 어떻게 되시나요? [SEP] 기준답변: 김영희 [SEP] 이전답변: ...,무응답,3


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)

train_dataset = Dataset.from_pandas(train_df[["text", "labels"]])
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "klue/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=160
    )

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

train_tokenized = train_tokenized.remove_columns(["text"])
test_tokenized = test_tokenized.remove_columns(["text"])

train_tokenized.set_format("torch")
test_tokenized.set_format("torch")

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/midas_recall_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.449821,1.342570,0.350000,0.129630
2,1.350107,1.340412,0.350000,0.129630
3,1.348047,1.343294,0.350000,0.129630
4,1.365677,1.321785,0.350000,0.129630
5,1.340454,1.299270,0.400000,0.259615
6,1.269697,1.219780,0.500000,0.395833
7,1.246017,1.190313,0.500000,0.395833
8,1.209689,1.175078,0.500000,0.395833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=80, training_loss=1.3121256172657012, metrics={'train_runtime': 231.6957, 'train_samples_per_second': 2.762, 'train_steps_per_second': 0.345, 'total_flos': 52623156019200.0, 'train_loss': 1.3121256172657012, 'epoch': 8.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 1.2194797992706299,
 'eval_accuracy': 0.5,
 'eval_macro_f1': 0.39583333333333337,
 'eval_runtime': 0.2833,
 'eval_samples_per_second': 70.597,
 'eval_steps_per_second': 10.59,
 'epoch': 8.0}

<<프로토타입 V3>>

In [ ]:
!pip install -q gradio openai pandas python-dotenv boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.5 MB/s eta 0:00:00


In [ ]:
!pip install -q gradio openai pandas

In [ ]:
import os
from openai import OpenAI

YNU_API_KEY = os.getenv("YNU_API_KEY")

client = OpenAI(
    api_key=YNU_API_KEY,
    base_url="https://factchat-cloud.mindlogic.ai/v1/gateway"
)

GPT_MODEL = "claude-sonnet-4-6"

response = client.chat.completions.create(
    model=GPT_MODEL,
    messages=[
        {"role": "user", "content": "안녕하세요. 한 문장으로 답해주세요."}
    ]
)

print(response.choices[0].message.content)

안녕하세요! 무엇을 도와드릴까요? 😊


In [ ]:
import json
import random
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
import gradio as gr


# =====================================================
# 1. DB 구조 기준 가짜 DB
# =====================================================

DB = {
    "users": [],
    "chat_sessions": [],
    "audio_records": [],
    "recall_questions": [],
    "risk_analysis_results": [],
    "recall_analysis_results": []
}

AUTO_ID = {
    "user_id": 1,
    "session_id": 1,
    "record_id": 1,
    "question_id": 1,
    "risk_result_id": 1,
    "recall_result_id": 1
}


def next_id(name):
    value = AUTO_ID[name]
    AUTO_ID[name] += 1
    return value


# =====================================================
# 2. 공통 함수
# =====================================================

def now_korea():
    return datetime.now(ZoneInfo("Asia/Seoul"))


def korean_weekday(dt):
    days = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]
    return days[dt.weekday()]


def normalize_text(text):
    if text is None:
        return ""

    text = str(text).strip()
    remove_words = [" ", ".", ",", "입니다", "이에요", "예요", "요", "네", "음", "어"]
    for w in remove_words:
        text = text.replace(w, "")

    return text


def is_no_answer(text):
    text = normalize_text(text)

    no_answer_words = [
        "모르겠",
        "기억안나",
        "기억이나지않",
        "생각안나",
        "잘몰라",
        "글쎄",
        "헷갈려",
        "몰라"
    ]

    return any(word in text for word in no_answer_words)


def simple_similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    if a in b or b in a:
        return 1.0

    set_a = set(a)
    set_b = set(b)

    common = set_a & set_b
    union = set_a | set_b

    return len(common) / max(len(union), 1)


def compare_answer(expected, current, question_type):
    if current is None or str(current).strip() == "":
        return "무응답", 20, 0.0

    if is_no_answer(current):
        return "무응답", 20, 0.0

    sim = simple_similarity(expected, current)

    if sim >= 0.75:
        return "일치", 0, sim
    elif sim >= 0.45:
        return "부분일치", 10, sim
    else:
        if question_type == "ORIENTATION":
            return "불일치", 20, sim
        return "불일치", 25, sim


def get_risk_level(score):
    if score >= 70:
        return "HIGH"
    elif score >= 40:
        return "MEDIUM"
    else:
        return "LOW"


def make_summary(profile_score, orientation_score, recall_score, no_answer_count, level):
    reasons = []

    if profile_score >= 10:
        reasons.append("개인정보 기반 질문에서 일부 불일치가 감지되었습니다")

    if orientation_score >= 10:
        reasons.append("날짜와 요일 등 지남력 질문에서 혼동이 감지되었습니다")

    if recall_score >= 10:
        reasons.append("기억회상 질문에서 이전 답변과 현재 답변의 불일치 또는 무응답이 감지되었습니다")

    if no_answer_count >= 2:
        reasons.append("무응답 또는 기억 회피 표현이 반복적으로 나타났습니다")

    if not reasons:
        return "현재 대화에서는 뚜렷한 인지 위험 신호가 감지되지 않았습니다."

    return f"{', '.join(reasons)}. 전체 위험도는 {level} 수준입니다."


# =====================================================
# 3. 사용자 / 세션 생성
# =====================================================

def parse_family_info(family_text):
    family = []

    lines = [line.strip() for line in family_text.split("\n") if line.strip()]

    for line in lines:
        if ":" in line:
            relation, name = line.split(":", 1)
        elif "：" in line:
            relation, name = line.split("：", 1)
        else:
            parts = line.split()
            if len(parts) >= 2:
                relation = parts[0]
                name = parts[1]
            else:
                continue

        family.append({
            "relation": relation.strip(),
            "name": name.strip()
        })

    return family


def create_user(name, birth_date, family_text):
    user_id = next_id("user_id")

    user = {
        "userId": user_id,
        "name": name.strip(),
        "birthDate": birth_date.strip(),
        "familyInfo": parse_family_info(family_text),
        "createdAt": now_korea().isoformat()
    }

    DB["users"].append(user)
    return user


def create_chat_session(user_id):
    session_id = next_id("session_id")

    session = {
        "sessionId": session_id,
        "userId": user_id,
        "startedAt": now_korea().isoformat(),
        "endedAt": None,
        "status": "ACTIVE"
    }

    DB["chat_sessions"].append(session)
    return session


def end_chat_session(session_id):
    for session in DB["chat_sessions"]:
        if session["sessionId"] == session_id:
            session["endedAt"] = now_korea().isoformat()
            session["status"] = "ENDED"
            return session

    return None


# =====================================================
# 4. recall_questions / audio_records 저장
# =====================================================

def create_recall_question(question_type, question_text, expected_answer=None, source="SYSTEM"):
    question_id = next_id("question_id")

    question = {
        "recallQuestionId": question_id,
        "questionType": question_type,
        "questionText": question_text,
        "expectedAnswer": expected_answer,
        "source": source,
        "createdAt": now_korea().isoformat()
    }

    DB["recall_questions"].append(question)
    return question


def save_audio_record(
    session_id,
    user_id,
    text,
    speaker,
    recall_question_id=None,
    answer_role=None,
    parent_record_id=None,
    audio_url=None
):
    record_id = next_id("record_id")

    record = {
        "recordId": record_id,
        "sessionId": session_id,
        "userId": user_id,
        "speaker": speaker,
        "text": text,
        "audioUrl": audio_url,
        "recallQuestionId": recall_question_id,
        "answerRole": answer_role,
        "parentRecordId": parent_record_id,
        "createdAt": now_korea().isoformat()
    }

    DB["audio_records"].append(record)
    return record


# =====================================================
# 5. GPT 호출
# =====================================================

def call_gpt(prompt, fallback):
    try:
        response = client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "너는 고령자를 대상으로 친절하게 대화하는 인지 위험도 추적 앱의 AI 대화자다."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.7
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        print("GPT 호출 오류:", e)
        return fallback


def gpt_generate_question(stage, user, memory_facts=None):
    if memory_facts is None:
        memory_facts = []

    family_text = ", ".join([f"{f['relation']} {f['name']}" for f in user["familyInfo"]])
    if not family_text:
        family_text = "가족 정보 없음"

    prompt = f"""
너는 고령자를 대상으로 하는 대화형 인지 위험도 추적 앱의 AI다.

사용자 정보:
- 이름: {user['name']}
- 생년월일: {user['birthDate']}
- 가족 정보: {family_text}

현재 단계:
{stage}

이전 기억 정보:
{json.dumps(memory_facts, ensure_ascii=False, indent=2)}

규칙:
- 한국어로 말한다.
- 질문은 한 번에 하나만 한다.
- 너무 길게 말하지 않는다.
- 진단처럼 말하지 않는다.
- 고령자에게 부드럽고 자연스럽게 질문한다.

단계 설명:
PROFILE_NAME: 이름 확인
PROFILE_BIRTH: 생년월일 확인
PROFILE_FAMILY: 가족 이름 확인
ORIENTATION_DATE: 오늘 날짜 확인
ORIENTATION_DAY: 오늘 요일 확인
MEMORY_LEARN: 이용자에 대해 알아가기 위한 질문
MEMORY_RECALL: 앞에서 사용자가 말한 내용을 다시 떠올리게 하는 질문

현재 단계에 맞는 질문 문장 하나만 출력해라.
"""

    fallback_map = {
        "PROFILE_NAME": "성함이 어떻게 되시나요?",
        "PROFILE_BIRTH": "생년월일이 어떻게 되시나요?",
        "PROFILE_FAMILY": "가족분 중 한 분의 성함을 말씀해주실 수 있을까요?",
        "ORIENTATION_DATE": "오늘은 몇 월 며칠인지 아시나요?",
        "ORIENTATION_DAY": "오늘은 무슨 요일인지 아시나요?",
        "MEMORY_LEARN": "요즘 자주 드시는 음식이나 좋아하시는 음식이 있으신가요?",
        "MEMORY_RECALL": "아까 말씀해주신 내용을 다시 떠올려보면 무엇이라고 하셨나요?"
    }

    return call_gpt(prompt, fallback_map.get(stage, "편하게 말씀해주실 수 있을까요?"))


def extract_memory_label(question, answer):
    prompt = f"""
다음 대화에서 나중에 회상 질문에 사용할 수 있는 기억 정보를 추출해라.

질문:
{question}

답변:
{answer}

반드시 JSON만 출력해라.

형식:
{{
  "label": "기억 항목 이름",
  "value": "사용자가 말한 핵심 답변",
  "recall_question": "나중에 다시 물어볼 회상 질문"
}}

예:
{{
  "label": "좋아하는 음식",
  "value": "김치찌개",
  "recall_question": "아까 좋아한다고 말씀하신 음식이 무엇이었나요?"
}}
"""

    fallback = json.dumps({
        "label": "대화 기억",
        "value": answer,
        "recall_question": "아까 말씀해주신 내용을 다시 떠올려보면 무엇이라고 하셨나요?"
    }, ensure_ascii=False)

    raw = call_gpt(prompt, fallback)

    try:
        start = raw.find("{")
        end = raw.rfind("}") + 1
        return json.loads(raw[start:end])
    except Exception:
        return {
            "label": "대화 기억",
            "value": answer,
            "recall_question": "아까 말씀해주신 내용을 다시 떠올려보면 무엇이라고 하셨나요?"
        }


# =====================================================
# 6. 대화 플로우 구성
# =====================================================

def build_flow(user):
    today = now_korea()
    date_answer = f"{today.month}월 {today.day}일"
    day_answer = korean_weekday(today)

    family = user["familyInfo"]

    if family:
        target = family[0]
        family_question = f"{target['relation']}분의 성함이 어떻게 되시나요?"
        family_answer = target["name"]
    else:
        family_question = "가족분 중 가장 자주 연락하는 분은 누구인가요?"
        family_answer = ""

    return [
        {
            "stage": "PROFILE_NAME",
            "type": "PROFILE",
            "answerRole": "PROFILE",
            "expectedAnswer": user["name"],
            "fixedQuestion": "성함이 어떻게 되시나요?"
        },
        {
            "stage": "PROFILE_BIRTH",
            "type": "PROFILE",
            "answerRole": "PROFILE",
            "expectedAnswer": user["birthDate"],
            "fixedQuestion": "생년월일이 어떻게 되시나요?"
        },
        {
            "stage": "PROFILE_FAMILY",
            "type": "PROFILE",
            "answerRole": "PROFILE",
            "expectedAnswer": family_answer,
            "fixedQuestion": family_question
        },
        {
            "stage": "ORIENTATION_DATE",
            "type": "ORIENTATION",
            "answerRole": "ORIENTATION",
            "expectedAnswer": date_answer,
            "fixedQuestion": "오늘은 몇 월 며칠인지 아시나요?"
        },
        {
            "stage": "ORIENTATION_DAY",
            "type": "ORIENTATION",
            "answerRole": "ORIENTATION",
            "expectedAnswer": day_answer,
            "fixedQuestion": "오늘은 무슨 요일인지 아시나요?"
        },
        {
            "stage": "MEMORY_LEARN",
            "type": "MEMORY_LEARN",
            "answerRole": "PAST",
            "expectedAnswer": None,
            "fixedQuestion": None
        },
        {
            "stage": "MEMORY_LEARN",
            "type": "MEMORY_LEARN",
            "answerRole": "PAST",
            "expectedAnswer": None,
            "fixedQuestion": None
        },
        {
            "stage": "MEMORY_LEARN",
            "type": "MEMORY_LEARN",
            "answerRole": "PAST",
            "expectedAnswer": None,
            "fixedQuestion": None
        },
        {
            "stage": "MEMORY_RECALL",
            "type": "RECALL",
            "answerRole": "CURRENT",
            "expectedAnswer": None,
            "fixedQuestion": None
        }
    ]


# =====================================================
# 7. 세션 분석
# =====================================================

def analyze_session(session_id):
    user_records = [
        r for r in DB["audio_records"]
        if r["sessionId"] == session_id and r["speaker"] == "USER"
    ]

    profile_score = 0
    orientation_score = 0
    recall_score = 0
    no_answer_count = 0
    question_results = []

    for record in user_records:
        qid = record["recallQuestionId"]

        if qid is None:
            continue

        question = next(
            (q for q in DB["recall_questions"] if q["recallQuestionId"] == qid),
            None
        )

        if question is None:
            continue

        qtype = question["questionType"]
        expected = question.get("expectedAnswer")

        if qtype == "MEMORY_LEARN":
            continue

        if expected is None or expected == "":
            continue

        result, point, sim = compare_answer(expected, record["text"], qtype)

        if result == "무응답":
            no_answer_count += 1

        if qtype == "PROFILE":
            profile_score += point
        elif qtype == "ORIENTATION":
            orientation_score += point
        elif qtype == "RECALL":
            recall_score += point

        question_results.append({
            "recordId": record["recordId"],
            "questionId": qid,
            "questionType": qtype,
            "question": question["questionText"],
            "expectedAnswer": expected,
            "currentAnswer": record["text"],
            "result": result,
            "similarity": round(sim, 3),
            "riskPoint": point
        })

    total_score = min(profile_score + orientation_score + recall_score, 100)
    level = get_risk_level(total_score)

    summary = make_summary(
        profile_score,
        orientation_score,
        recall_score,
        no_answer_count,
        level
    )

    result = {
        "riskResultId": next_id("risk_result_id"),
        "sessionId": session_id,
        "riskScore": total_score,
        "riskLevel": level,
        "summary": summary,
        "details": {
            "profileScore": profile_score,
            "orientationScore": orientation_score,
            "recallConsistencyScore": recall_score,
            "noAnswerCount": no_answer_count
        },
        "questionResults": question_results,
        "createdAt": now_korea().isoformat()
    }

    DB["risk_analysis_results"].append(result)
    return result


# =====================================================
# 8. Gradio 함수
# =====================================================

def init_app(name, birth_date, family_text):
    if not name.strip() or not birth_date.strip():
        return [], None, "이름과 생년월일은 반드시 입력해주세요.", ""

    user = create_user(name, birth_date, family_text)
    session = create_chat_session(user["userId"])
    flow = build_flow(user)

    state = {
        "user": user,
        "session": session,
        "flow": flow,
        "flowIndex": 0,
        "memoryFacts": [],
        "lastQuestion": None,
        "lastQuestionId": None,
        "lastBotRecordId": None,
        "ended": False
    }

    first_step = flow[0]
    question_text = first_step["fixedQuestion"] or gpt_generate_question(first_step["stage"], user)

    question = create_recall_question(
        question_type=first_step["type"],
        question_text=question_text,
        expected_answer=first_step["expectedAnswer"],
        source="SYSTEM"
    )

    bot_record = save_audio_record(
        session_id=session["sessionId"],
        user_id=user["userId"],
        text=question_text,
        speaker="AI",
        recall_question_id=question["recallQuestionId"],
        answer_role="QUESTION"
    )

    state["lastQuestion"] = first_step
    state["lastQuestionId"] = question["recallQuestionId"]
    state["lastBotRecordId"] = bot_record["recordId"]

    chat = [
        {"role": "assistant", "content": question_text}
    ]

    status = f"""
세션 시작 완료

userId: {user['userId']}
sessionId: {session['sessionId']}

현재 구조:
- chat_sessions: 대화 1회
- audio_records: 발화 1개
- recall_questions: 질문 목록
- risk_analysis_results: 최종 위험도 결과

종료하려면 대화창에 '대화 종료'라고 입력하세요.
"""

    return chat, state, status, ""


def chat_turn(user_message, chat_history, state):
    if state is None:
        return chat_history, state, "", "먼저 개인정보를 입력하고 세션 시작을 눌러주세요."

    if state.get("ended"):
        return chat_history, state, "", "이미 종료된 세션입니다."

    if user_message is None or user_message.strip() == "":
        return chat_history, state, "", "답변을 입력해주세요."

    user = state["user"]
    session = state["session"]

    chat_history.append({"role": "user", "content": user_message})

    if user_message.strip() == "대화 종료":
        end_chat_session(session["sessionId"])
        result = analyze_session(session["sessionId"])
        state["ended"] = True

        chat_history.append({
            "role": "assistant",
            "content": "대화를 종료하겠습니다. 지금까지의 대화 기반 분석 결과를 정리했습니다."
        })

        return chat_history, state, "", json.dumps(result, ensure_ascii=False, indent=2)

    last_question = state["lastQuestion"]
    last_qid = state["lastQuestionId"]

    user_record = save_audio_record(
        session_id=session["sessionId"],
        user_id=user["userId"],
        text=user_message,
        speaker="USER",
        recall_question_id=last_qid,
        answer_role=last_question["answerRole"],
        parent_record_id=state["lastBotRecordId"],
        audio_url=None
    )

    if last_question["type"] == "MEMORY_LEARN":
        question = next(
            (q for q in DB["recall_questions"] if q["recallQuestionId"] == last_qid),
            None
        )

        if question:
            memory = extract_memory_label(question["questionText"], user_message)
            memory["sourceRecordId"] = user_record["recordId"]
            state["memoryFacts"].append(memory)

    state["flowIndex"] += 1

    if state["flowIndex"] >= len(state["flow"]):
        end_chat_session(session["sessionId"])
        result = analyze_session(session["sessionId"])
        state["ended"] = True

        chat_history.append({
            "role": "assistant",
            "content": "준비된 질문이 모두 끝났습니다. 대화를 종료하고 분석 결과를 정리했습니다."
        })

        return chat_history, state, "", json.dumps(result, ensure_ascii=False, indent=2)

    next_step = state["flow"][state["flowIndex"]]

    if next_step["type"] == "RECALL":
        if state["memoryFacts"]:
            memory = random.choice(state["memoryFacts"])
            question_text = memory["recall_question"]
            expected_answer = memory["value"]
        else:
            question_text = "아까 말씀해주신 내용 중 기억나는 것을 다시 말씀해주실 수 있을까요?"
            expected_answer = ""
    else:
        question_text = next_step["fixedQuestion"] or gpt_generate_question(
            next_step["stage"],
            user,
            state["memoryFacts"]
        )
        expected_answer = next_step["expectedAnswer"]

    source = "GPT" if next_step["fixedQuestion"] is None else "SYSTEM"

    question = create_recall_question(
        question_type=next_step["type"],
        question_text=question_text,
        expected_answer=expected_answer,
        source=source
    )

    bot_record = save_audio_record(
        session_id=session["sessionId"],
        user_id=user["userId"],
        text=question_text,
        speaker="AI",
        recall_question_id=question["recallQuestionId"],
        answer_role="QUESTION"
    )

    state["lastQuestion"] = next_step
    state["lastQuestionId"] = question["recallQuestionId"]
    state["lastBotRecordId"] = bot_record["recordId"]

    chat_history.append({
        "role": "assistant",
        "content": question_text
    })

    return chat_history, state, "", "대화 진행 중입니다."


def show_table(table_name):
    data = DB.get(table_name, [])

    if not data:
        return pd.DataFrame()

    return pd.DataFrame(data)


def show_latest_result():
    if not DB["risk_analysis_results"]:
        return "아직 분석 결과가 없습니다. 대화 종료 후 확인할 수 있습니다."

    return json.dumps(DB["risk_analysis_results"][-1], ensure_ascii=False, indent=2)


# =====================================================
# 9. Gradio UI
# =====================================================

with gr.Blocks(title="MIDAS AI 기억회상 대화형 프로토타입") as demo:
    gr.Markdown("""
# MIDAS AI 기억회상 대화형 프로토타입

이 프로토타입은 음성 데이터 수집 전 단계이므로 채팅 입력을 사용합니다.

실제 서비스 연결 시:
- 현재 `text` 입력값 = STT 결과
- 현재 `audioUrl = None`
- 추후 AWS S3 음성 파일 경로가 `audioUrl`에 저장됨
- AI 분석 결과는 `risk_analysis_results`에 `sessionId` 기준으로 저장됨
""")

    state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 1. 개인정보 입력")

            name_input = gr.Textbox(
                label="이름",
                value="김영희"
            )

            birth_input = gr.Textbox(
                label="생년월일",
                value="1950년 3월 12일"
            )

            family_input = gr.Textbox(
                label="가족 인적사항",
                lines=5,
                value="장남: 김민수\n장녀: 김민지\n배우자: 박철수"
            )

            start_btn = gr.Button("세션 시작", variant="primary")

            status_box = gr.Textbox(
                label="상태",
                lines=12
            )

        with gr.Column(scale=2):
            gr.Markdown("## 2. AI 대화창")

            chatbot = gr.Chatbot(
                label="AI 대화",
                height=500,
                type="messages"
            )

            msg = gr.Textbox(
                label="사용자 답변 입력",
                placeholder="답변을 입력하세요. 종료하려면 '대화 종료'라고 입력하세요."
            )

            send_btn = gr.Button("전송")

            result_json = gr.Code(
                label="최종 위험도 분석 JSON",
                language="json"
            )

    gr.Markdown("## 3. 가짜 DB 테이블 확인")

    with gr.Row():
        table_selector = gr.Dropdown(
            choices=[
                "users",
                "chat_sessions",
                "audio_records",
                "recall_questions",
                "risk_analysis_results",
                "recall_analysis_results"
            ],
            value="audio_records",
            label="확인할 테이블"
        )

        refresh_btn = gr.Button("테이블 새로고침")
        latest_btn = gr.Button("최신 분석 결과 보기")

    table_output = gr.Dataframe(label="DB Table Preview")

    start_btn.click(
        init_app,
        inputs=[name_input, birth_input, family_input],
        outputs=[chatbot, state, status_box, result_json]
    )

    send_btn.click(
        chat_turn,
        inputs=[msg, chatbot, state],
        outputs=[chatbot, state, msg, result_json]
    )

    msg.submit(
        chat_turn,
        inputs=[msg, chatbot, state],
        outputs=[chatbot, state, msg, result_json]
    )

    refresh_btn.click(
        show_table,
        inputs=[table_selector],
        outputs=[table_output]
    )

    latest_btn.click(
        show_latest_result,
        inputs=[],
        outputs=[result_json]
    )

demo.launch(share=True)

/tmp/ipykernel_1419/3224514670.py:824: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b922ea071851afa22.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
